# Llama-server bench results

## Output schema (one file per sweep invocation)

Each sweep run writes `tests/results/llama-server-bench/<model>_<timestamp>.jsonl`. Every row is one measurement: the integer-`point` rows form the per-launch decay curve, and the per-launch summary fields (`pp_tps`, `pp_auc`, `gen_tps`, `wall_request_s`, `n_tokens_prompt`, …) are stamped on **every** point row for that launch — so the last point row of a launch carries the whole-prompt summary. There is no separate summary row. Failed launches write a single row (no point fields).

The sweep is a **four-phase chain** (like `llama-bench-tuner.py`, but each phase runs ONLY its previous winners, so the sweep stays small):

- **Phase 1** (`phase: "quant"`): every quant in `quants` at the *fixed* `server_parameters` (ctx-size, threads, device, cache dtypes), with the server's **default split-mode / flash-attn / batch / ubatch** (none of those flags passed), spec-OFF. Winner = the quant with the highest average `pp_auc`.
- **Phase 2** (`phase: "server"`): the `server_sweep_parameters` cells (flash-attn × split-mode, ...) on top of the fixed params, at the **phase-1 winning quant**, spec-off. Winner = the cell with the highest average `pp_auc`; its **complete** flag set (fixed + swept) is inherited by the later phases.
- **Phase 3** (`phase: "batch"`): `pp_parameters` cells (batch/ubatch, ...) at the **phase-1 + phase-2 winners**, still spec-off and pure PP (`ntg: 1`). Winner = the pp cell with the highest average `pp_auc`.
- **Phase 4** (`phase: "spec"`): every `spec_variants` entry (first = the spec-off control) at the **phase-1 + phase-2 + phase-3 winners**, generating `spec_ntg` tokens. Winner = the variant with the highest average `pp_auc` (a guard — the control winning is expected); per-variant `gen_tps` and the naive accept estimate (variant/control − 1, upper bound) are reference output.

**`pp_auc`** = the "area under the curve" of a launch's PP decay curve = the **sum of the per-segment marginal rates** (`tps_seg`, t/s). It is the winner metric of all four phases. **Every condition in every phase is run with EVERY prompt version**, and winners are the average across those launches. The winner picks and rankings are in the sibling `.summary.json` under `phases`.

| Field | Description |
|---|---|
| `phase` | `"quant"` (phase 1), `"server"` (phase 2), `"batch"` (phase 3), or `"spec"` (phase 4, the spec/MTP scan) |
| `run` | 1-based repeat index (from `--runs N`) |
| `spec` | Spec/MTP variant label — `"spec-off"` for every phase-1/2/3 row; the `spec_variants` labels in phase 4 (`spec-off` = control) |
| `launch` | 0-based index of this launch within the invocation (increments across all phases) |
| `model` | The `model_name` from the config |
| `quant` | The quant tag this launch ran with (from `model_base` + `quants`; file = `<base>-<quant>.gguf`) |
| `prompt` | Basename of the prompt file used by this launch (the "version") |
| `batch_size` | `--batch-size` value this launch ran with — **absent in phases 1-2** (server default was used) |
| `ubatch_size` | `--ubatch-size` value this launch ran with — **absent in phases 1-2** (server default was used) |
| `flash_attn`, `split_mode` | present only in phase 2+ rows (swept in phase 2 / inherited) |
| `ctx_size`, `threads`, `threads_batch`, `n_gpu_layers`, `cache_type_k`, `cache_type_v`, ... | the fixed server params, tagged on every row |
| (all other cell flags) | every cell flag is tagged the same way — phase-2/3/4 rows carry the full combined server + pp flag set |
| `status` | `"ok"`, `"error"` (launch failed — see `error`; driver errors also carry `server_log_tail`), or `"no_output"` (server ran but no progress lines were parsed from its log) |
| `n_points` | Number of progress points parsed from the server log (how full the decay curve is) |
| `n_tokens_prompt` | Prompt tokens the server actually processed (ground truth for context occupancy) |
| `pp_tps` | Final summary PP rate from the log's `prompt eval time` line (tokens/s over the whole prompt) |
| `pp_ms_per_tok` | Final summary PP latency from the same line (ms per token over the whole prompt) |
| `pp_auc` | **The winner metric**: sum of the per-segment `tps_seg` values for this launch (t/s-sum over the decay curve) |
| `gen_tps` | Final summary generation rate from the `eval time` line (tokens/s; a single-token estimate in phases 1-3 at `ntg: 1`, a real decode rate in phase 4 at `spec_ntg`) |
| `gen_ms_per_tok` | Final summary generation latency from the `eval time` line (ms per token) |
| `prompt_tokens_http` | `usage.prompt_tokens` from the `/v1/completions` response (should equal `n_tokens_prompt`) |
| `completion_tokens_http` | `usage.completion_tokens` from the response (≈ `ntg` / `spec_ntg`) |
| `wall_request_s` | Host-side wall clock around the HTTP request (seconds); excludes model load, which happens before the request |
| `point` | Progress-point index within the launch (integer 1..N) |
| `n_tokens_cum` | Cumulative prompt tokens at this point (the occupancy — x-axis of the decay curve) |
| `n_tokens_seg` | Tokens added between the previous point and this one (~2048-4096, smaller at the tail) |
| `t_cum_s` | Cumulative seconds the server reported at this point |
| `t_seg_s` | Seconds for just this segment (diff of cumulative times) |
| `tps_cum` | Running-average PP rate: `n_tokens_cum / t_cum_s` |
| `tps_seg` | Marginal PP rate: `n_tokens_seg / t_seg_s` — **the decay curve** to plot vs `n_tokens_cum` |

PP curves exist in **phase 1 + 2 + 3 rows** (`df[df["phase"].isin(["quant", "server", "batch"])]`); phase-4 rows are the spec scan (decode-rate measurements + a PP guard, not part of the PP-parameter sweep). For a per-launch pivot, take the max-`point` row per condition (it carries the whole-prompt summary, incl. `pp_auc`). Filter `status == "ok"` first.

In [2]:
import itertools

import matplotlib.pyplot as plt
import pandas as pd

In [ ]:
df = pd.read_json('./results/llama-server-bench/Qwen3.8-27B_2026-11-14_16-28-41.jsonl', lines=True)
df = df[df['status'] == 'ok']
df.head().T

,0,1,2,3,4
phase,server,server,server,server,server
run,1,1,1,1,1
spec,spec-off,spec-off,spec-off,spec-off,spec-off
launch,0,0,0,0,0
model,Qwen3.8-27B,Qwen3.8-27B,Qwen3.8-27B,Qwen3.8-27B,Qwen3.8-27B
quant,UD-Q4_K_M,UD-Q4_K_M,UD-Q4_K_M,UD-Q4_K_M,UD-Q4_K_M
prompt,qwen3_8_27b_v1.txt,qwen3_8_27b_v1.txt,qwen3_8_27b_v1.txt,qwen3_8_27b_v1.txt,qwen3_8_27b_v1.txt
ctx_size,131072,131072,131072,131072,131072
threads,12,12,12,12,12
threads_batch,12,12,12,12,12


In [5]:
df['flash_attn'].value_counts()

flash_attn
on     785
off     12
Name: count, dtype: int64

In [6]:
df['split_mode'].value_counts()

split_mode
layer     453
tensor    344
Name: count, dtype: int64

In [7]:
df['prompt'].value_counts()

prompt
qwen3_8_27b_v1.txt    314
qwen3_8_27b_v2.txt    267
qwen3_8_27b_v3.txt    216
Name: count, dtype: int64

In [ ]:
ok = df[df['status'] == 'ok']

# Phase 1 (quant) rows have no flash-attn/split-mode/batch/ubatch tags, and
# phase 3 (batch) rows inherit the phase-2 server cell -- build the condition
# label from the tags that exist on each row
def cond_label(r):
    bits = [f"{c}={r[c]}" for c in
            ('quant', 'flash_attn', 'split_mode', 'batch_size', 'ubatch_size')
            if c in r.index and r[c] is not None and not pd.isna(r[c])]
    return ' '.join(bits)

pp = ok[ok['phase'].isin(['quant', 'server', 'batch'])]   # PP curves in phases 1-3

plt.title('Prompt preprocessing rate vs occupancy')
for (phase, label), series_df in pp.groupby(['phase', pp.apply(cond_label, axis=1)]):
    if series_df.empty:
        continue
    # drop the first point (warm-up ramp) and the last (partial) segment
    plt.scatter(series_df['n_tokens_cum'][1:-2], series_df['tps_seg'][1:-2],
                label=f'[{phase}] {label}')

plt.xlabel('Cumulative Tokens')
plt.ylabel('Throughput (tps_seg)')
plt.legend(fontsize='small')
plt.show()